In [ ]:
import bs4
import requests
import pandas as pd
import os
from urllib.parse import urljoin
import re
import copy

def getInfoBox(soup, url):
    table = soup.select_one('table.infobox')
    rows = table.find_all('tr')

    data = []
    current_header = "General"

    for row in rows:
        # Check if this row is an infobox header
        header_cell = row.find('th', class_='infobox-header')
        if header_cell:
            current_header = header_cell.get_text(strip=True)
            continue
        
        th = row.find('th')
        td = row.find('td', class_='infobox-data')
        
        if th and td:
            label = th.get_text(strip=True)
            
            list_items = td.find_all('li')
            containers = list_items if list_items else [td]
            
            values_list = []
            links_list = []
            citations_list = []
            
            for container_orig in containers:
                # Make a deep copy to avoid destroying citations in the original soup object
                # if this cell is executed multiple times!
                container = copy.copy(container_orig)
                
                # Extract citations: Look for sup tags that contain bracketed numbers/letters or have class reference
                cites = []
                for sup in container.find_all('sup'):
                    cls = sup.get('class', [])
                    if 'reference' in cls or 'cite_ref' in cls or (sup.get_text() and re.match(r'^\[\w+\]$', sup.get_text().strip())):
                        cites.append(sup.get_text(strip=True))
                        sup.decompose()
                
                # Extract links, ignoring units (heuristic: ignoring short strings or numbers)
                lnks = []
                for a in container.find_all('a'):
                    link_text = a.get_text(strip=True)
                    link_href = a.get('href', '')
                    # Simple heuristic to exclude typical unit/symbol links
                    if len(link_text) > 2 and not any(char.isdigit() for char in link_text):
                        if link_href:
                            full_url = urljoin(url, link_href)
                            lnks.append(full_url)
                
                # Re-get the value after decomposing citations in the temporary copy
                # Also remove parenthetical text which usually contains redundant conversions/units
                value_clean = container.get_text(separator=' ', strip=True)
                value_clean = re.sub(r'\s*\([^)]*\)', '', value_clean).strip()
                
                # If after stripping units it becomes empty, we might want to skip it, 
                # but we shouldn't lose the citations if it had any.
                # We can merge its citations into the previous item in the list.
                if not value_clean:
                    if cites and citations_list:
                        # Append these citations to the previous item's citations
                        prev_cites = citations_list[-1]
                        if prev_cites:
                            citations_list[-1] = prev_cites + ", " + ", ".join(cites)
                        else:
                            citations_list[-1] = ", ".join(cites)
                    continue
                    
                values_list.append(value_clean)
                links_list.append(", ".join(lnks) if lnks else None)
                citations_list.append(", ".join(cites) if cites else None)
                
            if not values_list:
                continue
            
            value_out = values_list if list_items else (values_list[0] if values_list else None)
            links_out = links_list if list_items else (links_list[0] if links_list else None)
            cites_out = citations_list if list_items else (citations_list[0] if citations_list else None)
            
            data.append({
                'Category': current_header,
                'Label': label,
                'Value': value_out,
                'Links': links_out,
                'Citations': cites_out
            })

    df = pd.DataFrame(data)
    df.set_index(['Category', 'Label'], inplace=True)
    return df

def savePageAndInfoBox(url, file_name, data_dir):

    # make data dir
    os.makedirs(data_dir, exist_ok=True)
    
    # soup
    headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
    response = requests.get(url, headers=headers)
    soup = bs4.BeautifulSoup(response.text, 'html.parser')

    # save page
    with open(f'{data_dir}/{file_name}_wiki.html', 'w') as f:
        f.write(soup.prettify())
    
    # save infoBox
    infoBox = getInfoBox(soup, url)
    infoBox.to_csv(f'{data_dir}/{file_name}_infoBox_wiki.csv')

#savePageAndInfoBox("https://en.wikipedia.org/wiki/Solar_System", 'solarsystem', 'data/Sol')
savePageAndInfoBox("https://en.wikipedia.org/wiki/Sun", 'sun', 'data/Sol')
    

In [48]:
df

Value  \
Category                   Label                                                                                        
General                    Age                                                                    4.568 billion years   
                           Location                                 [Local Interstellar Cloud, Local Bubble, Orion...   
                           Nearest star                                            [Proxima Centauri, Alpha Centauri]   
Population                 Stars                                                                                  Sun   
                           Planets                                  [Mercury, Venus, Earth, Mars, Jupiter, Saturn,...   
                           Knowndwarf planets                       [Ceres, Orcus, Pluto, Haumea, Quaoar, Makemake...   
                           Knownnatural satellites                                                                758   
                           Knownminor planets                                                               1,462,402   
                           Knowncomets                                                                          4,629   
Planetary system           Star spectral type                                                                     G2V   
                           Frost line                                                                           ~5 AU   
                           Semi-major axisof outermost planet                                                30.07 AU   
                           Kuiper cliff                                                                      50–70 AU   
                           Heliopause                                                              detected at 120 AU   
                           Hill sphere                                                             178,000–227,000 AU   
Orbit aboutGalactic Center Invariable-to-galactic planeinclination                              ~60°, to the ecliptic   
                           Distance toGalactic Center                                                24,000–28,000 ly   
                           Orbital speed                                                                 720,000 km/h   
                           Orbital period                                                          ~230 million years   

                                                                                                                Links  \
Category                   Label                                                                                        
General                    Age                                                                                   None   
                           Location                                 [https://en.wikipedia.org/wiki/Local_Interstel...   
                           Nearest star                             [https://en.wikipedia.org/wiki/Proxima_Centaur...   
Population                 Stars                                                    https://en.wikipedia.org/wiki/Sun   
                           Planets                                  [https://en.wikipedia.org/wiki/Mercury_(planet...   
                           Knowndwarf planets                       [https://en.wikipedia.org/wiki/Ceres_(dwarf_pl...   
                           Knownnatural satellites                                                               None   
                           Knownminor planets                                                                    None   
                           Knowncomets                                                                           None   
Planetary system           Star spectral type                                                                    None   
                           Frost line                                                                            None 

# Planets

In [62]:
url = "https://en.wikipedia.org/wiki/Solar_System"
headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
response = requests.get(url, headers=headers)
soup = bs4.BeautifulSoup(response.text, 'html.parser')
infobox = getInfoBox(soup, url)
infobox.loc[('Population', 'Planets')]['Value']

['Mercury', 'Venus', 'Earth', 'Mars', 'Jupiter', 'Saturn', 'Uranus', 'Neptune']

In [ ]:
url = "https://en.wikipedia.org/wiki/Solar_System"
headers = {"User-Agent": "garrettstoll.info/1.0 (garrett@garrettstoll.info)"}
response = requests.get(url, headers=headers)
soup = bs4.BeautifulSoup(response.text, 'html.parser')

infobox = getInfoBox(soup, url)
links = infobox.loc[('Population', 'Planets')]['Links']
for i, planet in enumerate(infobox.loc[('Population', 'Planets')]['Value']):
    savePageAndInfoBox(links[i], planet.lower(), 'data/Sol/planets')

0
1
2
3
4
5
6
7
